In [1]:
import torch
import torchvision
import transformers

print(torch.__version__)
print(torchvision.__version__)
print(transformers.__version__)

2.13.0+cpu
0.28.0+cpu
5.15.0


In [2]:
from transformers import RTDetrImageProcessor
print("RT-DETR import OK")

RT-DETR import OK


In [3]:
from docling.document_converter import DocumentConverter
print("Docling import OK")

Docling import OK


In [ ]:
#                     YOUR DOC-LING UNIVERSE
#                              │
#              ┌───────────────┴───────────────┐
#              │                               │
#        CLASSIC PDF PIPELINE             VLM PIPELINE
#              │                               │
#       backend_options                  stage_model_specs
#              │                               │
#       layout_model_specs              Granite / Smol / etc.
#              │
#       pipeline_options
#              │
#       ┌──────┴─────────┐
#       │                │
#    Layout          Table Structure
#       │                │
#    Heron/Egret      TableFormer
#                      V1/V2
#                      │
#                cell matching
#                fast/accurate

In [ ]:
#                     DOCling PDF PARSING
#                            │
#              ┌─────────────┴─────────────┐
#              │                           │
#         HOW IT PARSES              WHAT IT PRODUCES
#              │                           │
#      ┌───────┼────────┐                 │
#      │       │        │                 │
#   Backend  Layout   Table            DoclingDocument
#                     structure             │
#                        │              ┌────┼────┐
#                        │              │    │    │
#                  TableFormer         tables text pages
#                  V1 / V2 / VLM
#                        │
#                 FAST / ACCURATE
#                        │
#                 cell matching

In [ ]:
#                  PDF
#                   ↓
#           PDF backend / text
#                   ↓
#              Layout model
#                   ↓
#           table bounding box
#                   ↓
#        ┌──────────┴───────────┐
#        │                      │
#    TableFormer V1        TableFormer V2
#        │                      │
#    text tokens +         image only
#    bboxes                + model bboxes
#        │                      │
#        └──────────┬───────────┘
#                   ↓
#              Table cells
#                   ↓
#              TableItem

In [27]:
#THIS IS CUDA GPU ACC CODE !!

import json
from pathlib import Path

from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)

from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureV2Options,
)

from docling.datamodel.base_models import InputFormat
# from docling.datamodel.accelerator_options import (
#     AcceleratorOptions,
#     AcceleratorDevice,
# )
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode

import pandas as pd

output_dir="docling_output"
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

def parse_pdf(pdf_path, output_dir="docling_output"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    opts = PdfPipelineOptions()
    opts.do_ocr = False
    opts.do_table_structure = True
    opts.table_structure_options.mode = TableFormerMode.ACCURATE
    opts.table_structure_options.do_cell_matching = True
    # opts.accelerator_options = AcceleratorOptions(
    #     device=AcceleratorDevice.CUDA,
    #     num_threads=8,
    # )

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=opts)
        }
    )

    result = converter.convert(pdf_path)
    doc = result.document
    name = Path(pdf_path).stem

    (output_dir / f"{name}.md").write_text(
        doc.export_to_markdown(), encoding="utf-8"
    )

    with open(output_dir / f"{name}.json", "w", encoding="utf-8") as f:
        json.dump(doc.export_to_dict(), f, indent=2, ensure_ascii=False)

    with pd.ExcelWriter(output_dir / f"{name}_tables.xlsx") as writer:
        for i, table in enumerate(doc.tables, 1):
            df = table.export_to_dataframe(doc=doc)
            df.to_excel(writer, sheet_name=f"Table_{i}", index=False)

    return result



def parse_pdf_v2(pdf_path, output_dir="docling_output"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    opts = PdfPipelineOptions()
    opts.do_ocr = False
    opts.do_table_structure = True
    opts.table_structure_options = TableStructureV2Options(
        do_cell_matching=True
    )

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=opts)
        }
    )

    result = converter.convert(pdf_path)
    doc = result.document
    name = Path(pdf_path).stem

    (output_dir / f"{name}_v2.md").write_text(
        doc.export_to_markdown(),
        encoding="utf-8"
    )

    with open(output_dir / f"{name}_v2.json", "w", encoding="utf-8") as f:
        json.dump(
            doc.export_to_dict(),
            f,
            indent=2,
            ensure_ascii=False
        )

    with pd.ExcelWriter(
        output_dir / f"{name}_v2_tables.xlsx"
    ) as writer:
        for i, table in enumerate(doc.tables, 1):
            df = table.export_to_dataframe(doc=doc)
            df.to_excel(
                writer,
                sheet_name=f"Table_{i}",
                index=False
            )

    return result


In [ ]:
pdf_path = r"D:\BSE_PDFS\500023.pdf"
# result = parse_pdf(pdf_path)
result = parse_pdf_v2(pdf_path)

print(type(result))
print(type(result.document))
print(type(result.input))

# print(result.document.tables)
# print(result.pages)
# print(result.timings)
# print(result.confidence)
# print(result.errors)


In [10]:
table = result.document.tables[0]

print(type(table.data))
print(table.data.model_dump().keys())

<class 'docling_core.types.doc.items.table.table_data.TableData'>
dict_keys(['table_cells', 'num_rows', 'num_cols', 'orientation', 'grid'])


In [11]:
print(table.data.model_dump())

{'table_cells': [{'bbox': {'l': 374.112, 't': 177.70799999999997, 'r': 417.6, 'b': 183.71399999999994, 'coord_origin': <CoordOrigin.TOPLEFT: 'TOPLEFT'>}, 'row_span': 1, 'col_span': 3, 'start_row_offset_idx': 0, 'end_row_offset_idx': 1, 'start_col_offset_idx': 2, 'end_col_offset_idx': 5, 'text': 'Quarter Ended', 'column_header': True, 'row_header': False, 'row_section': False, 'fillable': False}, {'bbox': {'l': 501.98400000000004, 't': 177.70799999999997, 'r': 535.3944, 'b': 183.71399999999994, 'coord_origin': <CoordOrigin.TOPLEFT: 'TOPLEFT'>}, 'row_span': 1, 'col_span': 1, 'start_row_offset_idx': 0, 'end_row_offset_idx': 1, 'start_col_offset_idx': 5, 'end_col_offset_idx': 6, 'text': 'Year Ended', 'column_header': True, 'row_header': False, 'row_section': False, 'fillable': False}, {'bbox': {'l': 82.36800000000005, 't': 189.668, 'r': 89.56806000000006, 'b': 196.67499999999995, 'coord_origin': <CoordOrigin.TOPLEFT: 'TOPLEFT'>}, 'row_span': 1, 'col_span': 1, 'start_row_offset_idx': 0, 'en

In [ ]:
for i, cell in enumerate(table.data.table_cells):
    print(
        i,
        repr(cell.text),
        "row:", cell.start_row_offset_idx,
        "col:", cell.start_col_offset_idx,
        "rowspan:", cell.row_span,
        "colspan:", cell.col_span,
        "col_header:", cell.column_header,
        "row_header:", cell.row_header,
    )

In [ ]:
df = table.export_to_dataframe(doc=result.document)

print(df.to_string())

In [15]:
for cell in table.data.table_cells:
    if cell.start_row_offset_idx in [15, 16, 17]:
        print(
            cell.start_row_offset_idx,
            cell.start_col_offset_idx,
            repr(cell.text),
            cell.bbox
        )

15 1 'Total Expenses' l=95.90299999999996 t=355.557 r=140.54292 b=362.564 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
15 2 '8,274.59' l=345.59899999999993 t=355.557 r=370.94460000000004 b=362.564 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
15 3 '9,761.51' l=396.2869999999999 t=355.557 r=421.34476000000006 b=362.564 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
15 4 '7,877.03' l=446.6869999999999 t=355.557 r=472.0326 b=362.564 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
15 5 '38,611.93' l=527.0389999999999 t=355.557 r=555.55217 b=362.564 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
17 0 '3' l=83.51899999999989 t=373.125 r=86.6869899999999 b=380.132 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
17 1 'ie (Loss) from ordinary activities before exceptional items and Tax' l=94.4629999999999 t=373.125 r=313.3430299999998 b=380.132 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
16 2 '(473.10)' l=348.7669999999998 t=373.125 r=372.95899999999983 b=380.132 coord_origin=<CoordOrigin.TOPLEFT:

In [16]:
print(table.data.model_dump().keys())

dict_keys(['table_cells', 'num_rows', 'num_cols', 'orientation', 'grid'])


In [17]:
print(table.data.grid)

[[TableCell(bbox=BoundingBox(l=82.36800000000005, t=189.668, r=89.56806000000006, b=196.67499999999995, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=0, end_col_offset_idx=1, text='St', column_header=True, row_header=False, row_section=False, fillable=False), TableCell(bbox=None, row_span=1, col_span=1, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=1, end_col_offset_idx=2, text='', column_header=False, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=374.112, t=177.70799999999997, r=417.6, b=183.71399999999994, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=3, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=2, end_col_offset_idx=5, text='Quarter Ended', column_header=True, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=374.112, t=177.70799999999997, r=417.6, b=183.

In [18]:
for i, row in enumerate(table.data.grid[14:19], start=14):
    print(i, row)
print("rows:", table.data.num_rows)
print("cols:", table.data.num_cols)

14 [TableCell(bbox=None, row_span=1, col_span=1, start_row_offset_idx=14, end_row_offset_idx=15, start_col_offset_idx=0, end_col_offset_idx=1, text='', column_header=False, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=94.46300000000002, t=346.47700000000003, r=215.4244400000001, b=352.483, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=14, end_row_offset_idx=15, start_col_offset_idx=1, end_col_offset_idx=2, text='e. Other Operating and General Expenses', column_header=False, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=348.47900000000004, t=346.47700000000003, r=371.23292000000015, b=352.483, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=14, end_row_offset_idx=15, start_col_offset_idx=2, end_col_offset_idx=3, text='3,257.73', column_header=False, row_header=False, row_section=False, fillable=False), TableCell(bbox=Bound

In [23]:
import pandas as pd
import numpy as np

def docling_tables_to_dfs(tables, y_tol=2):
    dfs = []

    for table in tables:
        cells = [
            c for c in table.data.table_cells
            if c.bbox is not None
        ]

        rows = []

        for cell in sorted(cells, key=lambda c: c.bbox.t):
            yc = (cell.bbox.t + cell.bbox.b) / 2

            for row in rows:
                if abs(yc - row["y"]) <= y_tol:
                    row["cells"].append(cell)
                    break
            else:
                rows.append({"y": yc, "cells": [cell]})

        data = [["" for _ in range(table.data.num_cols)] for _ in rows]

        for r, row in enumerate(rows):
            for cell in row["cells"]:
                c = cell.start_col_offset_idx
                span = cell.col_span

                for j in range(span):
                    if c + j < table.data.num_cols:
                        data[r][c + j] = cell.text.strip()

        dfs.append(pd.DataFrame(data))

    return dfs

def rebuild_table(table, row_factor=0.5):
    cells = [
        c for c in table.data.table_cells
        if c.bbox and c.text.strip()
    ]

    heights = [
        c.bbox.b - c.bbox.t
        for c in cells
    ]

    y_tol = np.mean(heights) * row_factor

    rows = []

    for cell in sorted(cells, key=lambda c: c.bbox.t):
        yc = (cell.bbox.t + cell.bbox.b) / 2

        for row in rows:
            if abs(yc - row["y"]) <= y_tol:
                row["cells"].append(cell)
                break
        else:
            rows.append({"y": yc, "cells": [cell]})

    data = [[""] * table.data.num_cols for _ in rows]

    for r, row in enumerate(rows):
        for cell in row["cells"]:
            c = cell.start_col_offset_idx

            for j in range(cell.col_span):
                if c + j < table.data.num_cols:
                    data[r][c + j] = cell.text.strip()

    return pd.DataFrame(data)

In [26]:
# dfs = docling_tables_to_dfs(result.document.tables)
dfs = [rebuild_table(table) for table in result.document.tables]

# for i, df in enumerate(dfs, 1):
#     print(f"\nTABLE {i}")
#     print(df)

with pd.ExcelWriter("tables.xlsx") as writer:
    for i, df in enumerate(dfs, 1):
        df.to_excel(writer, sheet_name=f"Table_{i}", index=False, header=False)